In [1]:
!pip install numpy torch torchsummary scikit-learn pandas plotly

In [2]:
# Importar bibliotecas necessárias

# Deep Learning / Machine Learning

import numpy as np
import torch
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# EDA
import pandas as pd
import plotly.express as px

## Carregar os dados

In [3]:
# Carregar o dataset
df_veiculos = pd.read_csv('data/veiculos.csv')

## Exploração Inicial dos dados

In [4]:
# Estrutura dos dados
df_veiculos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 835 entries, 0 to 834
Data columns (total 17 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Categoria                        835 non-null    int64  
 1   Cor                              835 non-null    object 
 2   Pais de Origem                   835 non-null    object 
 3   Ano Modelo                       835 non-null    int64  
 4   Ano Fabricação                   835 non-null    int64  
 5   Potencia                         835 non-null    int64  
 6   Quantidade de lugares            835 non-null    int64  
 7   Unico dono?                      835 non-null    int64  
 8   Ja teve sinistro?                835 non-null    int64  
 9   Ja foi carro de aplicativo?      835 non-null    int64  
 10  Revisoes em dia?                 835 non-null    int64  
 11  Sistema avancado de Multimidia?  835 non-null    int64  
 12  Tipo de Motorizacao   

In [5]:
# Visualizar primeiras linhas
df_veiculos.head()

,Categoria,Cor,Pais de Origem,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Tipo de Motorizacao,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
0,4,azul,Alemanha,2026,2025,143,4,0,0,1,1,1,Híbrido,60459,3,430,112898.39
1,7,verde,Alemanha,2024,2024,541,5,0,0,1,1,0,Híbrido,105982,7,484,887822.26
2,2,prata,Japão,2026,2025,94,4,0,1,1,0,0,Flex,38626,3,321,55516.43
3,4,preto,Japão,2022,2021,159,4,1,1,0,0,1,Flex,91185,2,415,147030.87
4,2,azul,Coreia do Sul,2026,2025,114,4,0,0,0,0,1,Flex,26037,3,381,93719.67


In [6]:
# Estatísticas
df_veiculos.describe()

,Categoria,Ano Modelo,Ano Fabricação,Potencia,Quantidade de lugares,Unico dono?,Ja teve sinistro?,Ja foi carro de aplicativo?,Revisoes em dia?,Sistema avancado de Multimidia?,Kilometragem,Tipo de Transmissao,Tamanho do porta malas,Valor de Venda
count,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,835.000000,8.350000e+02
mean,4.453892,2024.037126,2023.428743,307.476647,4.426347,0.492216,0.534132,0.462275,0.489820,0.489820,96439.767665,3.785629,395.049102,7.902733e+05
std,2.274606,1.402229,1.358211,320.806721,1.303114,0.500239,0.499133,0.498874,0.500196,0.500196,78575.237811,2.169836,131.769333,1.742400e+06
min,1.000000,2022.000000,2021.000000,70.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12611.000000,1.000000,80.000000,4.280000e+04
25%,2.000000,2023.000000,2022.000000,114.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,42289.500000,2.000000,308.000000,8.187166e+04
50%,4.000000,2024.000000,2024.000000,160.000000,4.000000,0.000000,1.000000,0.000000,0.000000,0.000000,70509.000000,3.000000,428.000000,1.436343e+05
75%,6.000000,2025.000000,2025.000000,371.500000,5.000000,1.000000,1.000000,1.000000,1.000000,1.000000,129393.500000,5.000000,490.500000,4.210006e+05
max,8.000000,2026.000000,2025.000000,1485.000000,7.000000,1.000000,1.000000,1.000000,1.000000,1.000000,403947.000000,9.000000,600.000000,9.433305e+06


In [7]:
# Mostrar os valores únicos das variáveis categoricas
for column in df_veiculos.select_dtypes(include=['object']).columns:
    print(f'{column}: {df_veiculos[column].unique()}')

Cor: ['azul' 'verde' 'prata' 'preto' 'cinza' 'vermelho' 'branco']
Pais de Origem: ['Alemanha' 'Japão' 'Coreia do Sul' 'Inglaterra' 'França' 'Itália'
 'Estados Unidos' 'China']
Tipo de Motorizacao: ['Híbrido' 'Flex' 'Elétrico' 'Gasolina']


In [8]:
"""Domínios
Categoria
1 - Econômico / Compacto
2 - Intermediário / Hatch Médio
3 - Sedan Compacto
4 - SUV de Entrada
5 - Sedan / SUV Médio
6 - Premium / Executivo
7 - Luxo / Superluxo
8 - Superesportivo / Hipercarro
Tipo de Transmissão
1 - Manual
2 - Automático
3 - CVT
4 - Automático 8 marchas
5 - Automático 10 marchas
6 - DCT
7 - Eletrônica
8 - Automático Esportivo
9 - Sequencial Paddle Shift
"""

'Domínios\nCategoria\n1 - Econômico / Compacto\n2 - Intermediário / Hatch Médio\n3 - Sedan Compacto\n4 - SUV de Entrada\n5 - Sedan / SUV Médio\n6 - Premium / Executivo\n7 - Luxo / Superluxo\n8 - Superesportivo / Hipercarro\nTipo de Transmissão\n1 - Manual\n2 - Automático\n3 - CVT\n4 - Automático 8 marchas\n5 - Automático 10 marchas\n6 - DCT\n7 - Eletrônica\n8 - Automático Esportivo\n9 - Sequencial Paddle Shift\n'

## Preparação de Dados para EDA

In [9]:
# Criar lista de varíaveis categóricas
categorical_features = df_veiculos.select_dtypes(include=['object']).columns.tolist()

# Incluir variáveis categoricas adicionais
additional_categorical_features = ['Categoria', 'Ano Modelo', 'Ano Fabricação', 'Unico dono?', 'Ja teve sinistro?', 'Ja foi carro de aplicativo?', 'Revisoes em dia?', 'Sistema avancado de Multimidia?', 'Tipo de Transmissao']
categorical_features.extend(additional_categorical_features)
categorical_features

['Cor',
 'Pais de Origem',
 'Tipo de Motorizacao',
 'Categoria',
 'Ano Modelo',
 'Ano Fabricação',
 'Unico dono?',
 'Ja teve sinistro?',
 'Ja foi carro de aplicativo?',
 'Revisoes em dia?',
 'Sistema avancado de Multimidia?',
 'Tipo de Transmissao']

In [10]:
# Criar uma lista de variáveis numéricas
numerical_features = df_veiculos.select_dtypes(include=['int64', 'float64']).columns.tolist()
# Remover o Target
numerical_features.remove('Valor de Venda')
# Remover features categoricas do tipo numérico
numerical_features = [feature for feature in numerical_features if feature not in categorical_features]
numerical_features

['Potencia', 'Quantidade de lugares', 'Kilometragem', 'Tamanho do porta malas']

In [11]:
# Variável target
target = ['Valor de Venda']

## EDA

In [12]:
# Distribuição da variável, target, usando plotly
fig = px.histogram(df_veiculos, x=target, nbins=50, title='Distribuição do Valor de Venda')
fig.show()

In [13]:
# Distribuição das variáveis numéricas
for feature in numerical_features:
    fig = px.histogram(df_veiculos, x=feature, nbins=50, title=f'Distribuição de {feature}')
    fig.show()

In [14]:
# BoxPlot das variáveis numéricas
for feature in numerical_features:
    fig = px.box(df_veiculos, y=feature, title=f'BoxPlot de {feature}')
    fig.show()

In [15]:
# Distribuição das variáveis categoricas
for feature in categorical_features:
    fig = px.histogram(df_veiculos, x=feature, title=f'Distribuição de {feature}')
    fig.show()

In [16]:
# Boxplot das variáveis categoricas com o target
for feature in categorical_features:
    fig = px.box(df_veiculos, x=feature, y=target, title=f'BoxPlot de {feature} com {target}')
    fig.show()